# Compare saved Beadform adapters on free Colab
Use an eligible free GPU and the same Google account that holds your adapters. This evaluates saved adapters, without retraining or paid services. It records strict JSON and original schema scores; these do not establish pattern quality. Defaults point to the two existing Beadform pilot runs. Choose a new output folder when changing the model, data, or code.


In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No free CUDA GPU offered. Stop and retry later; do not upgrade.")
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import subprocess,sys
repo=Path("/content/beadform")
if not repo.exists():
    subprocess.run(["git","clone","--depth","1","--branch","runpod-training-guide","https://github.com/c-tustin/beadform.git",str(repo)],check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-r",str(repo/"training/requirements.txt")],check=True)


In [ ]:
import hashlib, json, subprocess, sys, torch
from pathlib import Path
assert torch.cuda.is_available(), "No free CUDA GPU; stop and retry later."
repo = Path("/content/beadform")
drive_root = Path("/content/drive/MyDrive/Beadform-free-qwen")
run = drive_root/"runs"/"bounded-adapter-comparison-2026-10-17"
run.mkdir(parents=True, exist_ok=True)
data = repo/"training/benchmarks/schema-fresh-12.jsonl"
revision = subprocess.check_output(["git","rev-parse","HEAD"],cwd=repo,text=True).strip()
metadata = {"repo_commit":revision,"gpu":torch.cuda.get_device_name(0),
    "data_sha256":hashlib.sha256(data.read_bytes()).hexdigest(),
    "schema_sha256":hashlib.sha256((repo/"scene-schema.json").read_bytes()).hexdigest(),
    "note":"Same bounded configuration for both saved adapters. Structural benchmark only."}
metadata_path=run/"comparison-metadata.json"
if metadata_path.exists() and json.loads(metadata_path.read_text()) != metadata:
    raise RuntimeError("Configuration changed; use a new run directory.")
metadata_path.write_text(json.dumps(metadata,indent=2)+"\n")
freeze=subprocess.check_output([sys.executable,"-m","pip","freeze"],text=True)
(run/"pip-freeze.txt").write_text(freeze)
for label in ("pilot-02b","pilot-03-expanded"):
    adapter=drive_root/"runs"/label/"adapter"
    if not (adapter/"adapter_config.json").is_file():
        raise FileNotFoundError(f"Saved adapter missing: {adapter}")
    destination=run/label
    destination.mkdir(exist_ok=True)
    summary=destination/"summary.json"
    if summary.exists():
        print(label, "saved summary:", summary.read_text(),flush=True)
        continue
    command=[sys.executable,"-u",str(repo/"training/evaluate_lora.py"),
        "--data",str(data),"--output",str(destination/"predictions.jsonl"),
        "--summary",str(summary),"--adapter",str(adapter),
        "--bounded","--prompts-only","--resume"]
    print("START",label,flush=True)
    with (destination/"run.log").open("a") as log:
        process=subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
        for line in process.stdout:
            log.write(line); log.flush()
            print(line,end="",flush=True)
        code=process.wait()
    if code:
        raise RuntimeError(f"{label} exited with code {code}; inspect saved log.")
    print("COMPLETE",label,summary.read_text(),flush=True)
print("COMPARISON SAVED",run,flush=True)
